In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

print(os.listdir('/content/drive/MyDrive'))

['B.Tech. Semester 2 Timetable - Apr 2023 [Adm 2022] -V2.0 (2).pdf', 'It Lab10.gdoc', 'medical_certificate (2).pdf', 'medical_certificate (1).pdf', 'medical_certificate.pdf', 'medical_leave(18 & 19).pdf', ' IHUL.gdoc', 'Untitled document (15).gdoc', 'Untitled document (14).gdoc', 'DocScanner 15-Mar-2024 23-04.pdf', '1_intro_to_mgmt_1_2.pdf', 'Budgets.pdf', 'cs701-iom ful notes.pdf', 'Untitled document (13).gdoc', 'Untitled document (12).gdoc', 'Untitled document (11).gdoc', 'Untitled document (10).gdoc', 'ai_assignment.pdf', 'Untitled document (9).gdoc', 'Untitled document (8).gdoc', 'Untitled document (7).gdoc', 'Untitled document (6).gdoc', 'Document from 🙂.pdf', 'Untitled document (5).gdoc', 'Untitled document (4).gdoc', 'Untitled document (3).gdoc', 'Untitled document (2).gdoc', 'Untitled document (1).gdoc', 'Untitled document.gdoc', 'cloud_computing_assignment.pdf', 'VINAY_KUMAR_RESUME.pdf', 'DECLARATION_FORM_2022BCD0023.pdf', 'VINAY_RESUME (5).pdf', 'VINAY_RESUME (4).pdf', 'VINAY

In [5]:
import os

MODEL_PATH = "/content/drive/MyDrive/plag_models"

print("Files in plag_models:")
for item in os.listdir(MODEL_PATH):
    print(item)

Files in plag_models:
adapter_config.json
generation_config.json
config.json
tokenizer_config.json
deepseek-code-detector
AI_Text_Detector
ViT_Image_Detector


In [6]:
import os

TEXT_MODEL_PATH = "/content/drive/MyDrive/plag_models/AI_Text_Detector"

for root, dirs, files in os.walk(TEXT_MODEL_PATH):
    print(f"\nFolder: {root}")
    for file in files:
        print(f"  {file}")


Folder: /content/drive/MyDrive/plag_models/AI_Text_Detector
  tokenizer.json
  training_args.bin
  tokenizer_config.json
  config.json
  model.safetensors
  plag_(facebook_xlm_roberta_base).py


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/plag_models/AI_Text_Detector"

# Rename the weight file to the name Hugging Face expects if not already done
expected_weight_path = os.path.join(MODEL_PATH, "model.safetensors")
current_weight_path = os.path.join(MODEL_PATH, "text.safetensors")

if os.path.exists(current_weight_path) and not os.path.exists(expected_weight_path):
    print("Renaming text.safetensors to model.safetensors...")
    os.rename(current_weight_path, expected_weight_path)

print("Model files:", os.listdir(MODEL_PATH))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)
model.eval()

print("Text detector loaded successfully!")
print("Device:", device)
print("Labels:", model.config.id2label)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model files: ['tokenizer.json', 'training_args.bin', 'tokenizer_config.json', 'config.json', 'model.safetensors', 'plag_(facebook_xlm_roberta_base).py']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Text detector loaded successfully!
Device: cuda
Labels: {0: 'Human-written', 1: 'AI-generated'}


In [8]:
import torch

def predict_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    predicted_class = probs.argmax().item()
    confidence = probs[predicted_class].item() * 100

    print("Input:", text)
    print("Prediction:", model.config.id2label[predicted_class])
    print(f"Model confidence: {confidence:.2f}%")
    print(f"Human-written probability: {probs[0].item() * 100:.2f}%")
    print(f"AI-generated probability: {probs[1].item() * 100:.2f}%")

    return model.config.id2label[predicted_class]

In [9]:
!pip install -q datasets

In [10]:
from datasets import load_dataset

dataset = load_dataset(
    "Hello-SimpleAI/HC3",
    "default",
    revision="refs/convert/parquet"
)

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source'],
        num_rows: 48644
    })
})
{'id': '0', 'question': 'Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n\'t there only be one " # 1 " best seller ? Please explain like I\'m five.', 'human_answers': ['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence 

In [11]:

import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Select the test split
split = "test" if "test" in dataset else list(dataset.keys())[0]
examples = []

# 2. Prepare labeled examples
for row in dataset[split]:
    for text in row.get("human_answers", []) or []:
        if isinstance(text, str) and text.strip():
            examples.append((text.strip(), 0))

    for text in row.get("chatgpt_answers", []) or []:
        if isinstance(text, str) and text.strip():
            examples.append((text.strip(), 1))

df = pd.DataFrame(examples, columns=["text", "label"])
df = df.drop_duplicates(subset="text").reset_index(drop=True)

if df.empty:
    raise ValueError("No examples found. Check the dataset columns.")

print("Available unique samples:", len(df))
print("Class distribution:")
print(df["label"].value_counts())

# 3. Select 2,500 samples from each class
counts = df["label"].value_counts()

if counts.get(0, 0) < 2500 or counts.get(1, 0) < 2500:
    raise ValueError(
        "Not enough samples in one class to select 2,500 per class."
    )

df_test = (
    df.groupby("label", group_keys=False)
      .sample(n=2500, random_state=42)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

print("\nSelected samples:", len(df_test))
print(df_test["label"].value_counts())

# 4. Predict in batches
texts = df_test["text"].tolist()
y_true = df_test["label"].tolist()
y_pred = []

model.eval()
batch_size = 32

for start in tqdm(
    range(0, len(texts), batch_size),
    desc="Testing text detector"
):
    batch = texts[start:start + batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    with torch.inference_mode():
        logits = model(**inputs).logits
        predictions = logits.argmax(dim=-1)

    y_pred.extend(predictions.cpu().tolist())

# 5. Calculate metrics
accuracy = accuracy_score(y_true, y_pred)

print("\n" + "=" * 45)
print(f"TEXT DETECTOR ACCURACY: {accuracy * 100:.2f}%")
print("=" * 45)

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Human-written", "AI-generated"],
    digits=4,
    zero_division=0
))

print("Confusion Matrix (actual rows, predicted columns):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

# 6. Inspect predictions
df_test["predicted_label"] = y_pred
df_test["actual_class"] = df_test["label"].map({
    0: "Human-written",
    1: "AI-generated"
})
df_test["predicted_class"] = df_test["predicted_label"].map({
    0: "Human-written",
    1: "AI-generated"
})

display(df_test[[
    "text", "actual_class", "predicted_class"
]].head(10))


Available unique samples: 79331
Class distribution:
label
0    53086
1    26245
Name: count, dtype: int64

Selected samples: 5000
label
0    2500
1    2500
Name: count, dtype: int64


Testing text detector:   0%|          | 0/157 [00:00<?, ?it/s]


TEXT DETECTOR ACCURACY: 99.86%

Classification Report:
               precision    recall  f1-score   support

Human-written     0.9992    0.9980    0.9986      2500
 AI-generated     0.9980    0.9992    0.9986      2500

     accuracy                         0.9986      5000
    macro avg     0.9986    0.9986    0.9986      5000
 weighted avg     0.9986    0.9986    0.9986      5000

Confusion Matrix (actual rows, predicted columns):
[[2495    5]
 [   2 2498]]


,text,actual_class,predicted_class
0,They ca n't go straight west in an easterly wi...,Human-written,Human-written
1,There are a few different ways that electronic...,AI-generated,AI-generated
2,Calcium nitrate is an inorganic compound with ...,AI-generated,AI-generated
3,because the people who make the laws are dispr...,Human-written,Human-written
4,look up incongruity theory . the main idea is ...,Human-written,Human-written
5,Meteorologists actually have found duplicate s...,Human-written,Human-written
6,"You 're right , the distances ( assuming you '...",Human-written,Human-written
7,"In photogrammetry and computer stereo vision, ...",Human-written,Human-written
8,Do you have swelling on the face? The extracti...,Human-written,Human-written
9,Empty space . That 's why gasses need so much ...,Human-written,Human-written
